# Dimensionality Reduction: Singular Value Decomposition (SVD)

This notebook explores the mathematical engine behind data compression and latent feature extraction. We will cover:
1. **The Theoretical Framework:** Manually factorizing a matrix into $U$, $\Sigma$, and $V^T$ to perform image compression.
2. **Truncated SVD for Sparse Data:** Why SVD succeeds where PCA fails when dealing with sparse datasets like text (NLP) or recommendation systems.
3. **Latent Feature Interpretation:** Examining what the reduced components actually represent.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

sns.set_theme(style="whitegrid")

### 1. The Theory in Action: Image Compression using full SVD
SVD factorizes a matrix $X$ into $U \Sigma V^T$. 

Let's generate a high-resolution 2D mathematical pattern (an image) and use SVD to compress it. By keeping only the top $k$ singular values (the highest numbers in the $\Sigma$ diagonal), we can rebuild the image using a fraction of the original data.

In [ ]:
# Generate a complex 2D image (a 300x300 matrix)
np.random.seed(42)
x = np.linspace(-5, 5, 300)
y = np.linspace(-5, 5, 300)
X_grid, Y_grid = np.meshgrid(x, y)
image_matrix = np.sin(X_grid**2 + Y_grid**2) + np.random.normal(0, 0.1, (300, 300))

# Perform full SVD using NumPy
U, Sigma, VT = np.linalg.svd(image_matrix, full_matrices=False)

print(f"Original Image Shape: {image_matrix.shape}")
print(f"U matrix shape (Left Singular Vectors): {U.shape}")
print(f"Sigma array shape (Singular Values): {Sigma.shape}")
print(f"VT matrix shape (Right Singular Vectors): {VT.shape}")

# Reconstruct the image using different numbers of latent components (k)
def reconstruct_image(U, Sigma, VT, k):
    return np.dot(U[:, :k], np.dot(np.diag(Sigma[:k]), VT[:k, :]))

components_to_test = [5, 20, 50]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(image_matrix, cmap='gray')
axes[0].set_title("Original (300 Components)")
axes[0].axis('off')

for i, k in enumerate(components_to_test):
    compressed_img = reconstruct_image(U, Sigma, VT, k)
    axes[i+1].imshow(compressed_img, cmap='gray')
    axes[i+1].set_title(f"Reconstructed (k={k})")
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

print("--- DIAGNOSTIC INSIGHT ---")
print("Notice how at k=5, we only get the 'macro' structure (the overall X shape). ")
print("By k=50, the image is virtually indistinguishable from the original, yet we threw away ~83% of the data! ")
print("This proves that SVD filters out noise and isolates the true 'signal'.")

### 2. Truncated SVD for Sparse Matrices (NLP & Text)
PCA requires centering data (subtracting the mean). If you have a massive matrix of mostly zeros (like word counts in documents), subtracting the mean turns all those zeros into non-zeros, destroying your RAM.

**Truncated SVD does not center the data.** This makes it the industry standard for NLP (Latent Semantic Analysis) and Recommendation Systems.

In [ ]:
# 1. Create a corpus of documents
corpus = [
    "machine learning is fascinating",
    "artificial intelligence and machine learning",
    "deep learning neural networks",
    "the stock market is volatile today",
    "financial market trends and stocks",
    "investment strategies for the stock market"
]

# 2. Convert to a Sparse TF-IDF Matrix (Thousands of documents x Thousands of words)
vectorizer = TfidfVectorizer()
X_sparse = vectorizer.fit_transform(corpus)

print(f"Sparse Document-Term Matrix Shape: {X_sparse.shape} (Documents x Vocabulary Words)")
print("Notice that X_sparse is stored in a compressed sparse format to save memory.\n")

# 3. Apply Truncated SVD (Reducing down to 2 Latent 'Topics')
svd = TruncatedSVD(n_components=2, random_state=42)
X_latent = svd.fit_transform(X_sparse)

print(f"Reduced Latent Matrix Shape: {X_latent.shape} (Documents x Latent Topics)")

### 3. Interpreting Latent Features (Hidden Structure)
Let's map these documents into our new 2D space. Remember, SVD doesn't just crush data; it finds *hidden structures*.

In [ ]:
# Create a DataFrame for visualization
df_topics = pd.DataFrame(X_latent, columns=['Latent Topic 1', 'Latent Topic 2'])
df_topics['Document_Text'] = corpus

# Let's assume the first 3 are 'Tech' and the last 3 are 'Finance'
df_topics['True_Category'] = ['Technology', 'Technology', 'Technology', 'Finance', 'Finance', 'Finance']

plt.figure(figsize=(10, 6))
sns.scatterplot(x='Latent Topic 1', y='Latent Topic 2', hue='True_Category', data=df_topics, s=150, palette='Set1')

# Add text labels
for i in range(df_topics.shape[0]):
    plt.text(df_topics['Latent Topic 1'][i] + 0.02, df_topics['Latent Topic 2'][i], 
             f"Doc {i}", fontsize=10)

plt.title("Truncated SVD: Document Groupings by Latent Topics", fontsize=14)
plt.show()

print("--- THE FINAL TAKEAWAY ---")
print("SVD completely ignored the individual words and instead grouped the documents based on their underlying meaning (Latent Semantics).")
print("It mathematically isolated 'Technology' along one axis and 'Finance' along the other, proving its power to extract hidden signals from highly sparse, high-dimensional data.")